# Apprentissage Fédéré avec Fed-BioMed

**Simulation de 3 hôpitaux** sur une seule machine via Fed-BioMed v6.

## Prérequis (dans l'ordre)
1. Préparer les données   : `python scripts/prepare_data.py`
2. Enregistrer les datasets : `python scripts/register_datasets.py`
3. **Exécuter la cellule 1 ci-dessous** (démarre le researcher / serveur gRPC)
4. Ouvrir un terminal et lancer : `scripts\start_all_nodes.bat`
5. Attendre que les 3 nœuds soient connectés, puis continuer.

## Cellule 1 — Initialisation du Researcher (démarrer EN PREMIER)

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators import FedAverage

from modeles.lstm import BiLSTMTrainingPlan
from modeles.resnet import ResNet1DTrainingPlan
from modeles.inceptiontime import InceptionTimeTrainingPlan

# Répertoire de config du researcher (créé automatiquement)
RESEARCHER_CONFIG = os.path.join(PROJECT_ROOT, 'nodes_config', 'researcher')
os.makedirs(RESEARCHER_CONFIG, exist_ok=True)

TAG = 'gait_marche'   # tag déclaré dans register_datasets.py
N_ROUNDS = 10         # nombre de rounds FL
N_EPOCHS_LOCAL = 3    # epochs d'entraînement local par round et par nœud
BATCH_SIZE = 32

print('Researcher initialisé. Démarrez maintenant les nœuds avec scripts\\start_all_nodes.bat')

## Cellule 2 — Création de l'Expérience FL

> Attendez que les 3 nœuds soient connectés avant d'exécuter cette cellule.

In [ ]:
model_args = {
    'n_channels':  22,
    'hidden_size': 64,
    'num_layers':  2,
    'n_classes':   2,
    'dropout':     0.5,
}

training_args = {
    'loader_args': {'batch_size': BATCH_SIZE},
    'optimizer_args': {'lr': 1e-3},
    'epochs': N_EPOCHS_LOCAL,
    'dry_run': False,
    'batch_maxnum': 0,  # 0 = pas de limite
}

exp = Experiment(
    tags=[TAG],
    training_plan_class=BiLSTMTrainingPlan,
    model_args=model_args,
    training_args=training_args,
    round_limit=N_ROUNDS,
    aggregator=FedAverage(),
    config_path=RESEARCHER_CONFIG,
)

print(f'Expérience créée. Nœuds connectés : {list(exp.training_data().node_ids())}')

## Cellule 3 — Entraînement FL (BiLSTM + FedAvg)

In [ ]:
print(f'Démarrage de {N_ROUNDS} rounds FL (BiLSTM + FedAvg)...')
exp.run()
print('Entraînement FL terminé.')

## Cellule 4 — Résultats FL et comparaison avec le centralisé

In [ ]:
# Récupération des métriques par round
replies = exp.federated_params()
print(f'Paramètres fédérés disponibles pour {len(replies)} rounds')

# Modèle agrégé final
final_params = exp.aggregated_params()
print(f'Rounds effectués : {len(final_params)}')

# Sauvegarde du modèle fédéré final
os.makedirs(os.path.join(PROJECT_ROOT, 'resultats'), exist_ok=True)
if final_params:
    last_round = max(final_params.keys())
    last_params = final_params[last_round]['params']
    torch.save(last_params, os.path.join(PROJECT_ROOT, 'resultats', 'bilstm_federe.pt'))
    print(f'Modèle fédéré (round {last_round}) sauvegardé dans resultats/')

## Cellule 5 — Expérience SCAFFOLD (variante FedAvg avec correction de variance)

In [ ]:
from fedbiomed.researcher.aggregators import Scaffold

exp_scaffold = Experiment(
    tags=[TAG],
    training_plan_class=BiLSTMTrainingPlan,
    model_args=model_args,
    training_args=training_args,
    round_limit=N_ROUNDS,
    aggregator=Scaffold(),
    config_path=RESEARCHER_CONFIG,
)

print('Démarrage SCAFFOLD...')
exp_scaffold.run()
print('SCAFFOLD terminé.')

## Cellule 6 — Comparaison FedAvg vs SCAFFOLD vs Centralisé

In [ ]:
# Chargement des modèles centralisés pour comparaison
from modeles.lstm import BiLSTMNet

centralise_path = os.path.join(PROJECT_ROOT, 'resultats', 'bilstm_centralise.pt')
if os.path.exists(centralise_path):
    model_central = BiLSTMNet()
    model_central.load_state_dict(torch.load(centralise_path, map_location='cpu'))
    print('Modèle centralisé chargé.')
else:
    print('Modèle centralisé non trouvé — exécutez d\'abord le notebook 02.')

# Résumé des expériences
print('\n=== COMPARAISON FL vs CENTRALISÉ ===')
print('  Approche        | Accuracy Test')
print('  --------------- | -------------')
print('  Centralisé      | (voir notebook 02)')
print('  FedAvg          | (voir métriques ci-dessus)')
print('  SCAFFOLD        | (voir métriques ci-dessus)')
print('\nNote : pour l\'accuracy exacte sur le test set, évaluer le modèle agrégé')
print('sur les données de test centralisées (notebook 02 requis).')

## Cellule 7 — Expérience avec ResNet1D (comparaison architectures)

Optionnel — relance une expérience FL avec ResNet1D pour comparer les architectures.

In [ ]:
model_args_resnet = {'n_channels': 22, 'n_classes': 2}

exp_resnet = Experiment(
    tags=[TAG],
    training_plan_class=ResNet1DTrainingPlan,
    model_args=model_args_resnet,
    training_args=training_args,
    round_limit=N_ROUNDS,
    aggregator=FedAverage(),
    config_path=RESEARCHER_CONFIG,
)
print('Démarrage FL ResNet1D...')
exp_resnet.run()
print('Terminé.')